# Cocogoat 2026 - Task 2: VAE + latent world model (64x64 video)

Per-frame beta-VAE (ConvNeXt depthwise blocks, PixelUnshuffle with DC-AE shortcuts, bottleneck attention, EQ-VAE regulariser, adversarial fine-tuning) + a space-time transformer world model with EMA memory that predicts the next frame's latent. Two datasets: Colored Moving MNIST (controlled dynamics, the first rung of your ViVAE curriculum) and your `Shelterdataset.npz`. Budget: ~1.5 h per dataset. Outputs: `MyDrive/Cocogoat/runs/task2/<dataset>/{vae,wm}/`.

**How to run:** `Runtime -> Change runtime type -> G4 (RTX PRO 6000)`, set the budgets in the settings cell (5),
then `Runtime -> Run all`.
Keep this tab open (Colab Pro stops idle/closed sessions). Everything is written to
`MyDrive/Cocogoat/runs/...`; if Colab disconnects, just *Run all* again - training resumes from the last checkpoint.
You do not need to inspect results: metrics, sample grids and logs are saved to Drive for analysis.

In [ ]:
# 1) Mount Google Drive (outputs + datasets live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get / update the code (public repo, branch claude/cocogoat-revival-glfnmz)
import os, subprocess
REPO, BRANCH = "https://github.com/Martyn0324/Cocogoat.git", "claude/cocogoat-revival-glfnmz"
if not os.path.exists('/content/Cocogoat/.git'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, '/content/Cocogoat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Cocogoat', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Cocogoat', 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(subprocess.run(['git', '-C', '/content/Cocogoat', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
os.chdir('/content/Cocogoat')

In [ ]:
# 3) Extra packages (torch/torchvision/transformers come with Colab)
!pip -q install -r /content/Cocogoat/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Colab Secrets are only readable from the notebook kernel: export HF_TOKEN for the training subprocesses
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN exported')
except Exception as exc:
    print('No HF_TOKEN secret (only needed for gated Hugging Face datasets):', type(exc).__name__)
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
# 4) GPU smoke test (tiny models, a few steps, always from scratch)
!python -m cocogoat.tasks.task2_world --smoke --dataset mmnist --run_root /content/drive/MyDrive/Cocogoat/runs/task2_smoke --force

In [ ]:
# 5) Settings - edit freely. Training time budgets in hours, per dataset (evaluation/data time not included).
#    Quick test: small budgets AND a separate folder (e.g. '.../runs/task2_test'): a finished Task 2 stage is not
#    extended later, so a test inside runs/task2 would block the full run there.
VAE_HOURS, WM_HOURS = 0.75, 0.75
RUN_ROOT = '/content/drive/MyDrive/Cocogoat/runs/task2'
# World-model evaluation: EVAL_CONTEXT real frames are shown, then EVAL_HORIZON frames are generated (any length;
# clips are rolled out in batches). A Shelter clip must have CONTEXT + HORIZON frames without a scene cut.
EVAL_CONTEXT, EVAL_HORIZON = 64, 64
# Shelter: drop the black intro (the anime starts around frame 500); 0 keeps every frame.
SHELTER_START_FRAME = 500
EVAL = f'--eval_context {EVAL_CONTEXT} --eval_horizon {EVAL_HORIZON}'

In [ ]:
# 6) Colored Moving MNIST: VAE, then world model (resumes automatically after a disconnect)
!python -m cocogoat.tasks.task2_world --dataset mmnist --vae_hours {VAE_HOURS} --wm_hours {WM_HOURS} --run_root {RUN_ROOT} {EVAL}

In [ ]:
# 7) Shelter (your anime music-video frames): VAE, then world model
!python -m cocogoat.tasks.task2_world --dataset shelter --vae_hours {VAE_HOURS} --wm_hours {WM_HOURS} --run_root {RUN_ROOT} {EVAL} --start_frame {SHELTER_START_FRAME}

In [ ]:
# 8) Results
from IPython.display import Image, Markdown, display
for ds in ['mmnist', 'shelter']:
    run = f'{RUN_ROOT}/{ds}'
    try:
        display(Markdown(f'## {ds}\n' + open(f'{run}/wm/report.md').read()))
        display(Image(f'{run}/vae/samples/recon_final.png'))
        display(Image(f'{run}/wm/samples/rollout_final.png'))
    except FileNotFoundError as exc:
        print(ds, 'not finished:', exc)